# pyAstroTrader

# Create Model

After downloading the quotes data for the asset selected with the ASSET_TO_CALCULATE environment variable, we need to add the astrological data to the quotes and then generate a XGBoost model

First of all, we need to import the models that we need to process.

In [1]:
import os
import gc
import multiprocessing as mp
import pickle

import pandas as pd
import dask.dataframe as dd
from dask.multiprocessing import get
import numpy as np
import plotly.graph_objects as go

from sklearn.model_selection import KFold
from sklearn.model_selection import train_test_split as ttsplit
from sklearn.metrics import mean_squared_error as mse

import xgboost as xgb
from xgboost import XGBClassifier
from xgboost import plot_importance
from xgboost import plot_tree

from IPython.display import display, HTML

import eli5



```pyastrotrader``` is a python module that we created, in order to calculate astrological charts based on specific dates, and also to calculate aspects between charts

In [2]:
from pyastrotrader import calculate_chart, calculate_aspects, calculate_transits, get_degrees, get_degree
from pyastrotrader.utils import create_input_json
from pyastrotrader.constants import *

Import all settings and helper functions that we will use in the next cells

In [3]:
from settings import *
from helpers import *

USING_CACHED_DATAFRAME = False
CACHE_FILE = './output/{}.{}.cache'.format(ASSET_TO_CALCULATE, datetime.datetime.strftime(datetime.datetime.today(), '%Y%m%d') )
CACHE_ASTRO_COLUMNS = './output/{}.{}.astro.cache'.format(ASSET_TO_CALCULATE, datetime.datetime.strftime(datetime.datetime.today(), '%Y%m%d') )

if os.path.isfile(CACHE_FILE):
    USING_CACHED_DATAFRAME = True

Read the CSV file with the quotes downloaded, and also create a counter column to help in the calculated columns below

In [4]:
if not USING_CACHED_DATAFRAME:
    StockPrices = pd.read_csv("{}.csv".format(SOURCE_FILE))
    StockPrices['Counter'] = np.arange(len(StockPrices))

Using several helper functions from ```helpers.py``` module, for each day we need to determine several indicators like:
* The current trend
* the future trend
* If there was a change in the trend ( a swing trade opportunity )
* current volatility for the previous days
* and many other indicators

In [5]:
if not USING_CACHED_DATAFRAME:
    max_counter = StockPrices['Counter'].max()

    StockPricesDask = dd.from_pandas(StockPrices, npartitions=NPARTITIONS)
    StockPrices['CorrectedDate'] = StockPricesDask.map_partitions(lambda df : df.apply(lambda x : correct_date(x), axis =1)).compute(scheduler='threads')
    StockPrices['PreviousStartPrice'] = StockPricesDask.map_partitions(lambda df : df.apply(lambda x : get_previous_stock_price(StockPrices, x, SWING_TRADE_DURATION), axis =1), meta='float').compute(scheduler='threads')
    StockPrices['FutureFinalPrice'] = StockPricesDask.map_partitions(lambda df : df.apply(lambda x : get_future_stock_price(StockPrices, x, max_counter, SWING_TRADE_DURATION), axis =1 ), meta='float').compute(scheduler='threads')
    StockPrices['PreviousStartDate'] = StockPricesDask.map_partitions(lambda df : df.apply(lambda x : get_previous_stock_date(StockPrices, x, SWING_TRADE_DURATION), axis =1 ), meta='float').compute(scheduler='threads')
    StockPrices['FutureFinalDate'] = StockPricesDask.map_partitions(lambda df : df.apply(lambda x : get_future_stock_date(StockPrices, x, max_counter, SWING_TRADE_DURATION), axis =1 ), meta='float').compute(scheduler='threads')

    StockPricesDask = dd.from_pandas(StockPrices, npartitions=NPARTITIONS)
    StockPrices['CurrentTrend'] = StockPricesDask.map_partitions(lambda df : df.apply(lambda x : calculate_current_trend(x), axis = 1), meta='float').compute(scheduler='threads')

    StockPricesDask = dd.from_pandas(StockPrices, npartitions=NPARTITIONS)
    StockPrices['FutureTrend'] = StockPricesDask.map_partitions(lambda df : df.apply(lambda x : calculate_future_trend(x), axis = 1), meta='float').compute(scheduler='threads')

    StockPricesDask = dd.from_pandas(StockPrices, npartitions=NPARTITIONS)
    StockPrices['SwingStrength'] = StockPricesDask.map_partitions(lambda df : df.apply(lambda x : calculate_swing_strenght(x), axis =1), meta='float').compute(scheduler='threads')
    StockPrices['IntradayVolatility'] = StockPricesDask.map_partitions(lambda df : df.apply(lambda x : calculate_intraday_volatility(StockPrices, x, SWING_TRADE_DURATION), axis =1), meta='float').compute(scheduler='threads')

    StockPrices['FutureTrendMax'] = StockPricesDask.map_partitions(lambda df : df.apply(lambda x : get_future_stock_max_price(StockPrices, x, max_counter, SWING_TRADE_DURATION), axis = 1), meta='float').compute(scheduler='threads')
    StockPrices['FutureTrendMin'] = StockPricesDask.map_partitions(lambda df : df.apply(lambda x : get_future_stock_min_price(StockPrices, x, max_counter, SWING_TRADE_DURATION), axis = 1), meta='float').compute(scheduler='threads')

    StockPricesDask = dd.from_pandas(StockPrices, npartitions=NPARTITIONS)
    StockPrices['IsSwing'] = StockPricesDask.map_partitions(lambda df : df.apply(lambda x : detect_swing_trade(x, SWING_EXPECTED_VOLATILITY), axis =1), meta='float').compute(scheduler='threads')
    StockPricesDask = dd.from_pandas(StockPrices, npartitions=NPARTITIONS)
    StockPrices['IsSwing'] = StockPricesDask.map_partitions(lambda df : df.apply(lambda x : clean_swing_trade(StockPrices, x, SWING_EXPECTED_VOLATILITY), axis =1), meta='float').compute(scheduler='threads')

    StockPricesDask = dd.from_pandas(StockPrices, npartitions=NPARTITIONS)
    StockPrices['StockIncreasedPrice'] = StockPricesDask.map_partitions(lambda df : df.apply(lambda x : detect_price_increase(x, STAGNATION_THRESHOLD), axis =1), meta='float').compute(scheduler='threads')
    StockPrices['StockDecreasedPrice'] = StockPricesDask.map_partitions(lambda df : df.apply(lambda x : detect_price_decrease(x, STAGNATION_THRESHOLD), axis =1), meta='float').compute(scheduler='threads')
    StockPrices['StockStagnated'] = StockPricesDask.map_partitions(lambda df : df.apply(lambda x : detect_price_stagnated(x, STAGNATION_THRESHOLD), axis =1), meta='float').compute(scheduler='threads')
    StockPrices['StockPriceChange'] = StockPricesDask.map_partitions(lambda df : df.apply(lambda x : calculate_price_change(StockPrices, x), axis =1), meta='float').compute(scheduler='threads')

    StockPrices['TARGET_IS_SWING'] = StockPrices['IsSwing']
    StockPrices['TARGET_PRICE_INCREASE'] = StockPrices['StockIncreasedPrice']
    StockPrices['TARGET_PRICE_DECREASE'] = StockPrices['StockDecreasedPrice']
    StockPrices['TARGET_PRICE_STAGNATION'] = StockPrices['StockStagnated']
    StockPrices['TARGET_PRICE_CHANGE'] = StockPrices['StockPriceChange']                                                                


C:\Users\khali\OneDrive\Bureau\pyAstroTrader-master\.venv\Lib\site-packages\dask\dataframe\dask_expr\_expr.py:4140: FutureWarning: Meta is not valid, `map_partitions` and `map_overlap` expects output to be a pandas object. Try passing a pandas object as meta or a dict or tuple representing the (name, dtype) of the columns. In the future the meta you passed will not work.
  warnings.warn(
C:\Users\khali\OneDrive\Bureau\pyAstroTrader-master\notebooks\helpers.py:45: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  return float(df[df['Counter'] == (x['Counter'] - swing_trade_duration)]['Price'])
C:\Users\khali\OneDrive\Bureau\pyAstroTrader-master\.venv\Lib\site-packages\dask\dataframe\dask_expr\_expr.py:4140: FutureWarning: Meta is not valid, `map_partitions` and `map_overlap` expects output to be a pandas object. Try passing a pandas object as meta or a dict or tuple representing the (name, dty

C:\Users\khali\OneDrive\Bureau\pyAstroTrader-master\.venv\Lib\site-packages\dask\dataframe\dask_expr\_expr.py:4140: FutureWarning: Meta is not valid, `map_partitions` and `map_overlap` expects output to be a pandas object. Try passing a pandas object as meta or a dict or tuple representing the (name, dtype) of the columns. In the future the meta you passed will not work.
  warnings.warn(
C:\Users\khali\OneDrive\Bureau\pyAstroTrader-master\.venv\Lib\site-packages\dask\dataframe\dask_expr\_expr.py:4140: FutureWarning: Meta is not valid, `map_partitions` and `map_overlap` expects output to be a pandas object. Try passing a pandas object as meta or a dict or tuple representing the (name, dtype) of the columns. In the future the meta you passed will not work.
  warnings.warn(
C:\Users\khali\OneDrive\Bureau\pyAstroTrader-master\.venv\Lib\site-packages\dask\dataframe\dask_expr\_expr.py:4140: FutureWarning: Meta is not valid, `map_partitions` and `map_overlap` expects output to be a pandas obj

After all the analisys, we can generate a excel file in order to help debug the indicators generated above

In [6]:
if not USING_CACHED_DATAFRAME:
    output_excel_file='./output/{}.Analisys.xlsx'.format(ASSET_TO_CALCULATE)
    StockPrices.to_excel(output_excel_file)

To debug the indicators, we can plot a stock chart with the swing indication, but this is commented out as it requires a lot of computational resources.

In [7]:
"""
swing_to_chart = []
for index, current_swing in StockPrices[StockPrices['IsSwing'] == 1].iterrows():
    swing_to_chart.append(dict(
        x0=current_swing['CorrectedDate'], 
        x1=current_swing['CorrectedDate'], 
        y0=0, 
        y1=1, 
        xref='x', 
        yref='paper',
        line_width=2))
"""

"\nswing_to_chart = []\nfor index, current_swing in StockPrices[StockPrices['IsSwing'] == 1].iterrows():\n    swing_to_chart.append(dict(\n        x0=current_swing['CorrectedDate'], \n        x1=current_swing['CorrectedDate'], \n        y0=0, \n        y1=1, \n        xref='x', \n        yref='paper',\n        line_width=2))\n"

In [8]:
"""
fig = go.Figure(data=[go.Candlestick(
                x=StockPrices['CorrectedDate'],
                open=StockPrices['Open'],
                high=StockPrices['High'],
                low=StockPrices['Low'],
                close=StockPrices['Price'])])
fig.update_layout(
    title="{} Detected Swing Trade Opportunities".format(ASSET_TO_CALCULATE),
    width=1000,
    height=500,
    xaxis_rangeslider_visible=False,
    shapes=swing_to_chart,
    margin=go.layout.Margin(
        l=0,
        r=0,
        b=0,
        t=30,
        pad=4
    ),    
)
#fig.show()
"""

'\nfig = go.Figure(data=[go.Candlestick(\n                x=StockPrices[\'CorrectedDate\'],\n                open=StockPrices[\'Open\'],\n                high=StockPrices[\'High\'],\n                low=StockPrices[\'Low\'],\n                close=StockPrices[\'Price\'])])\nfig.update_layout(\n    title="{} Detected Swing Trade Opportunities".format(ASSET_TO_CALCULATE),\n    width=1000,\n    height=500,\n    xaxis_rangeslider_visible=False,\n    shapes=swing_to_chart,\n    margin=go.layout.Margin(\n        l=0,\n        r=0,\n        b=0,\n        t=30,\n        pad=4\n    ),    \n)\n#fig.show()\n'

Well, in order to calculate the astrological indicators for the current ASSET_TO_CALCULATE, we need to generate a natal chart of the asset, which traditionally is the first trade date on the current exchange 

In [9]:
if not USING_CACHED_DATAFRAME:
    asset_natal_chart_input = create_input_json(NATAL_DATE, 
                                            DEFAULT_PARAMETERS, 
                                            DEFAULT_CONFIG)

    asset_natal_chart = calculate_chart(asset_natal_chart_input)
    dates_to_generate = list(StockPrices['CorrectedDate'])

Now, for all the dates on the pandas dataframe containing the quotes, we need to generate astrological charts with the list of planets to consider: ```PLANETS_TO_CALCULATE```, their aspects: ```ASPECTS_TO_CALCULATE```

In [10]:
if not USING_CACHED_DATAFRAME:
    from functools import partial
    
    with mp.Pool(processes = NPARTITIONS) as p:
        results = p.map(partial(generate_charts, asset_natal_chart=asset_natal_chart), dates_to_generate)

    for x in results:
        charts[x[0]] = x[1]
        aspects[x[0]] = x[2]
        aspects_transiting[x[0]] = x[3]

We have the natal chart and also all the charts for each date in the pandas dataframe, now we need to add to the pandas dataframe, the astrological aspects that occur in each date, we will set only to 1 if there is a aspect occuring or 0 if not, we also will check for aspects on the transiting chart as well as aspects between the natal chart and the transiting chart

**astro_columns** will indicate the name of the columns containing astrological indicators in the pandas dataframe

In [11]:
if not USING_CACHED_DATAFRAME:
    astro_columns = []

    for current_planet in PLANETS_TO_CALCULATE:
        if current_planet != SATURN:
            column_name="ASTRO_{}_POSITION".format(PLANETS[current_planet]).upper()
            StockPricesDask = dd.from_pandas(StockPrices, npartitions=NPARTITIONS)
            StockPrices[column_name] = StockPricesDask.map_partitions(lambda df : df.apply(lambda x : int(get_degree_for_planet(x, current_planet) / 3), axis =1), meta='int').compute(scheduler='threads')
            StockPrices[column_name] = pd.to_numeric(StockPrices[column_name],  downcast='float', errors='coerce')   
            astro_columns.append(column_name)   
        for second_planet in PLANETS_TO_CALCULATE:
            if current_planet == second_planet:
                continue
            
            column_name="ASTRO_{}_{}_DIFF".format(PLANETS[current_planet], PLANETS[second_planet]).upper()
            StockPricesDask = dd.from_pandas(StockPrices, npartitions=NPARTITIONS)
            StockPrices[column_name] = StockPricesDask.map_partitions(lambda df : df.apply(lambda x : int(int(get_degree_for_planet(x, current_planet) - get_degree_for_planet(x, second_planet))/ 3), axis =1), meta='int').compute(scheduler='threads')
            StockPrices[column_name] = pd.to_numeric(StockPrices[column_name],  downcast='float', errors='coerce')   
            astro_columns.append(column_name)   
            
            column_name="ASTRO_{}_{}_DIFF_ABS".format(PLANETS[current_planet], PLANETS[second_planet]).upper()
            StockPricesDask = dd.from_pandas(StockPrices, npartitions=NPARTITIONS)
            StockPrices[column_name] = StockPricesDask.map_partitions(lambda df : df.apply(lambda x : abs(int(get_degree_for_planet(x, current_planet) - get_degree_for_planet(x, second_planet))/ 3), axis =1), meta='int').compute(scheduler='threads')
            StockPrices[column_name] = pd.to_numeric(StockPrices[column_name],  downcast='float', errors='coerce')   
            astro_columns.append(column_name)   
        

    for first_planet in PLANETS_TO_CALCULATE:
        for second_planet in PLANETS_TO_CALCULATE:
            for aspect in ASPECTS_TO_CALCULATE:
                column_name="ASTRO_{}_{}_{}".format(PLANETS[first_planet],ASPECT_NAME[aspect],PLANETS[second_planet]).upper()
                aspect_column_name = column_name
                astro_columns.append(column_name)
                StockPricesDask = dd.from_pandas(StockPrices, npartitions=NPARTITIONS)
                StockPrices[column_name] = StockPricesDask.map_partitions(lambda df : df.apply(lambda x : is_aspected(x, first_planet, second_planet, aspect), axis =1), meta='float').compute(scheduler='threads')
                StockPrices[column_name] = pd.to_numeric(StockPrices[column_name],  downcast='float', errors='coerce')

                StockPricesDask = dd.from_pandas(StockPrices, npartitions=NPARTITIONS)
                column_name="ASTRO_TRANSITING_{}_{}_{}".format(PLANETS[first_planet],ASPECT_NAME[aspect],PLANETS[second_planet]).upper()
                astro_columns.append(column_name)
                StockPrices[column_name] = StockPricesDask.map_partitions(lambda df : df.apply(lambda x : is_aspected_transiting(x, first_planet, second_planet, aspect), axis =1), meta='float').compute(scheduler='threads')
                StockPrices[column_name] = pd.to_numeric(StockPrices[column_name],  downcast='float', errors='coerce')                 


C:\Users\khali\OneDrive\Bureau\pyAstroTrader-master\.venv\Lib\site-packages\dask\dataframe\dask_expr\_expr.py:4140: FutureWarning: Meta is not valid, `map_partitions` and `map_overlap` expects output to be a pandas object. Try passing a pandas object as meta or a dict or tuple representing the (name, dtype) of the columns. In the future the meta you passed will not work.
  warnings.warn(
C:\Users\khali\OneDrive\Bureau\pyAstroTrader-master\.venv\Lib\site-packages\dask\dataframe\dask_expr\_expr.py:4140: FutureWarning: Meta is not valid, `map_partitions` and `map_overlap` expects output to be a pandas object. Try passing a pandas object as meta or a dict or tuple representing the (name, dtype) of the columns. In the future the meta you passed will not work.
  warnings.warn(
C:\Users\khali\OneDrive\Bureau\pyAstroTrader-master\.venv\Lib\site-packages\dask\dataframe\dask_expr\_expr.py:4140: FutureWarning: Meta is not valid, `map_partitions` and `map_overlap` expects output to be a pandas obj

C:\Users\khali\OneDrive\Bureau\pyAstroTrader-master\.venv\Lib\site-packages\dask\dataframe\dask_expr\_expr.py:4140: FutureWarning: Meta is not valid, `map_partitions` and `map_overlap` expects output to be a pandas object. Try passing a pandas object as meta or a dict or tuple representing the (name, dtype) of the columns. In the future the meta you passed will not work.
  warnings.warn(
C:\Users\khali\OneDrive\Bureau\pyAstroTrader-master\.venv\Lib\site-packages\dask\dataframe\dask_expr\_expr.py:4140: FutureWarning: Meta is not valid, `map_partitions` and `map_overlap` expects output to be a pandas object. Try passing a pandas object as meta or a dict or tuple representing the (name, dtype) of the columns. In the future the meta you passed will not work.
  warnings.warn(
C:\Users\khali\OneDrive\Bureau\pyAstroTrader-master\.venv\Lib\site-packages\dask\dataframe\dask_expr\_expr.py:4140: FutureWarning: Meta is not valid, `map_partitions` and `map_overlap` expects output to be a pandas obj

C:\Users\khali\OneDrive\Bureau\pyAstroTrader-master\.venv\Lib\site-packages\dask\dataframe\dask_expr\_expr.py:4140: FutureWarning: Meta is not valid, `map_partitions` and `map_overlap` expects output to be a pandas object. Try passing a pandas object as meta or a dict or tuple representing the (name, dtype) of the columns. In the future the meta you passed will not work.
  warnings.warn(
C:\Users\khali\OneDrive\Bureau\pyAstroTrader-master\.venv\Lib\site-packages\dask\dataframe\dask_expr\_expr.py:4140: FutureWarning: Meta is not valid, `map_partitions` and `map_overlap` expects output to be a pandas object. Try passing a pandas object as meta or a dict or tuple representing the (name, dtype) of the columns. In the future the meta you passed will not work.
  warnings.warn(
C:\Users\khali\OneDrive\Bureau\pyAstroTrader-master\.venv\Lib\site-packages\dask\dataframe\dask_expr\_expr.py:4140: FutureWarning: Meta is not valid, `map_partitions` and `map_overlap` expects output to be a pandas obj

C:\Users\khali\OneDrive\Bureau\pyAstroTrader-master\.venv\Lib\site-packages\dask\dataframe\dask_expr\_expr.py:4140: FutureWarning: Meta is not valid, `map_partitions` and `map_overlap` expects output to be a pandas object. Try passing a pandas object as meta or a dict or tuple representing the (name, dtype) of the columns. In the future the meta you passed will not work.
  warnings.warn(
C:\Users\khali\OneDrive\Bureau\pyAstroTrader-master\.venv\Lib\site-packages\dask\dataframe\dask_expr\_expr.py:4140: FutureWarning: Meta is not valid, `map_partitions` and `map_overlap` expects output to be a pandas object. Try passing a pandas object as meta or a dict or tuple representing the (name, dtype) of the columns. In the future the meta you passed will not work.
  warnings.warn(
C:\Users\khali\OneDrive\Bureau\pyAstroTrader-master\.venv\Lib\site-packages\dask\dataframe\dask_expr\_expr.py:4140: FutureWarning: Meta is not valid, `map_partitions` and `map_overlap` expects output to be a pandas obj

C:\Users\khali\OneDrive\Bureau\pyAstroTrader-master\.venv\Lib\site-packages\dask\dataframe\dask_expr\_expr.py:4140: FutureWarning: Meta is not valid, `map_partitions` and `map_overlap` expects output to be a pandas object. Try passing a pandas object as meta or a dict or tuple representing the (name, dtype) of the columns. In the future the meta you passed will not work.
  warnings.warn(
C:\Users\khali\OneDrive\Bureau\pyAstroTrader-master\.venv\Lib\site-packages\dask\dataframe\dask_expr\_expr.py:4140: FutureWarning: Meta is not valid, `map_partitions` and `map_overlap` expects output to be a pandas object. Try passing a pandas object as meta or a dict or tuple representing the (name, dtype) of the columns. In the future the meta you passed will not work.
  warnings.warn(
C:\Users\khali\OneDrive\Bureau\pyAstroTrader-master\.venv\Lib\site-packages\dask\dataframe\dask_expr\_expr.py:4140: FutureWarning: Meta is not valid, `map_partitions` and `map_overlap` expects output to be a pandas obj

C:\Users\khali\OneDrive\Bureau\pyAstroTrader-master\.venv\Lib\site-packages\dask\dataframe\dask_expr\_expr.py:4140: FutureWarning: Meta is not valid, `map_partitions` and `map_overlap` expects output to be a pandas object. Try passing a pandas object as meta or a dict or tuple representing the (name, dtype) of the columns. In the future the meta you passed will not work.
  warnings.warn(
C:\Users\khali\OneDrive\Bureau\pyAstroTrader-master\.venv\Lib\site-packages\dask\dataframe\dask_expr\_expr.py:4140: FutureWarning: Meta is not valid, `map_partitions` and `map_overlap` expects output to be a pandas object. Try passing a pandas object as meta or a dict or tuple representing the (name, dtype) of the columns. In the future the meta you passed will not work.
  warnings.warn(
C:\Users\khali\OneDrive\Bureau\pyAstroTrader-master\.venv\Lib\site-packages\dask\dataframe\dask_expr\_expr.py:4140: FutureWarning: Meta is not valid, `map_partitions` and `map_overlap` expects output to be a pandas obj

C:\Users\khali\OneDrive\Bureau\pyAstroTrader-master\.venv\Lib\site-packages\dask\dataframe\dask_expr\_expr.py:4140: FutureWarning: Meta is not valid, `map_partitions` and `map_overlap` expects output to be a pandas object. Try passing a pandas object as meta or a dict or tuple representing the (name, dtype) of the columns. In the future the meta you passed will not work.
  warnings.warn(
C:\Users\khali\OneDrive\Bureau\pyAstroTrader-master\.venv\Lib\site-packages\dask\dataframe\dask_expr\_expr.py:4140: FutureWarning: Meta is not valid, `map_partitions` and `map_overlap` expects output to be a pandas object. Try passing a pandas object as meta or a dict or tuple representing the (name, dtype) of the columns. In the future the meta you passed will not work.
  warnings.warn(
C:\Users\khali\OneDrive\Bureau\pyAstroTrader-master\.venv\Lib\site-packages\dask\dataframe\dask_expr\_expr.py:4140: FutureWarning: Meta is not valid, `map_partitions` and `map_overlap` expects output to be a pandas obj

C:\Users\khali\OneDrive\Bureau\pyAstroTrader-master\.venv\Lib\site-packages\dask\dataframe\dask_expr\_expr.py:4140: FutureWarning: Meta is not valid, `map_partitions` and `map_overlap` expects output to be a pandas object. Try passing a pandas object as meta or a dict or tuple representing the (name, dtype) of the columns. In the future the meta you passed will not work.
  warnings.warn(
C:\Users\khali\AppData\Local\Temp\ipykernel_23252\3978573733.py:23: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  StockPrices[column_name] = StockPricesDask.map_partitions(lambda df : df.apply(lambda x : abs(int(get_degree_for_planet(x, current_planet) - get_degree_for_planet(x, second_planet))/ 3), axis =1), meta='int').compute(scheduler='threads')
C:\Users\khali\OneDrive\

C:\Users\khali\OneDrive\Bureau\pyAstroTrader-master\.venv\Lib\site-packages\dask\dataframe\dask_expr\_expr.py:4140: FutureWarning: Meta is not valid, `map_partitions` and `map_overlap` expects output to be a pandas object. Try passing a pandas object as meta or a dict or tuple representing the (name, dtype) of the columns. In the future the meta you passed will not work.
  warnings.warn(
C:\Users\khali\AppData\Local\Temp\ipykernel_23252\3978573733.py:23: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  StockPrices[column_name] = StockPricesDask.map_partitions(lambda df : df.apply(lambda x : abs(int(get_degree_for_planet(x, current_planet) - get_degree_for_planet(x, second_planet))/ 3), axis =1), meta='int').compute(scheduler='threads')
C:\Users\khali\OneDrive\

We need also to determine which planets are retrograde in each date of the pandas dataframe

In [12]:
if not USING_CACHED_DATAFRAME:
    for first_planet in []:
        column_name="ASTRO_{}_RETROGADE".format(PLANETS[first_planet]).upper()
        astro_columns.append(column_name)
        StockPricesDask = dd.from_pandas(StockPrices, npartitions=NPARTITIONS)
        StockPrices[column_name] = StockPricesDask.map_partitions(lambda df : df.apply(lambda x : is_retrograde(x, first_planet), axis =1), meta='float').compute(scheduler='threads')
        StockPrices[column_name] = pd.to_numeric(StockPrices[column_name],  downcast='float',errors='coerce')
        
if USING_CACHED_DATAFRAME:        
    StockPrices = pd.read_pickle(CACHE_FILE)
    with open(CACHE_ASTRO_COLUMNS, 'rb') as f:
        astro_columns = pickle.load(f)    
else:
    StockPrices.to_pickle(CACHE_FILE)
    with open(CACHE_ASTRO_COLUMNS, 'wb') as f:
        pickle.dump(astro_columns, f)

After the pandas dataframe has been populated with the astrological indicators, we can now train the XGBoost models in order to predict the following target variables:
* Price Increase: There is a increase in price after that date
* Price Decrease: There is a decrease in price after that date
* Price Stagnation: There is a stagnation in price after that date
* Swing Trade: There is a change in trend after that date
    
**Important to notice that we will use as input only the astro_columns columns which contains astrological indicators**

In [13]:
booster_price_change, score_price_change = get_best_booster('TARGET_PRICE_CHANGE', MAX_INTERACTIONS, StockPrices, astro_columns)
print("Best Score for Price Change Model:{}".format(score_price_change))

C:\Users\khali\OneDrive\Bureau\pyAstroTrader-master\.venv\Lib\site-packages\xgboost\training.py:199: UserWarning: [11:27:36] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "silent" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


TARGET_PRICE_CHANGE - 0 of 50, 0.17037253081798553


C:\Users\khali\OneDrive\Bureau\pyAstroTrader-master\.venv\Lib\site-packages\xgboost\training.py:199: UserWarning: [11:27:37] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "silent" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


TARGET_PRICE_CHANGE - 1 of 50, 0.04049249738454819


C:\Users\khali\OneDrive\Bureau\pyAstroTrader-master\.venv\Lib\site-packages\xgboost\training.py:199: UserWarning: [11:27:37] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "silent" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


...

TARGET_PRICE_CHANGE - 10 of 50, 8.474028618365992e-06


C:\Users\khali\OneDrive\Bureau\pyAstroTrader-master\.venv\Lib\site-packages\xgboost\training.py:199: UserWarning: [11:27:42] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "silent" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


TARGET_PRICE_CHANGE - 11 of 50, 8.474028618365992e-06


C:\Users\khali\OneDrive\Bureau\pyAstroTrader-master\.venv\Lib\site-packages\xgboost\training.py:199: UserWarning: [11:27:43] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "silent" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


TARGET_PRICE_CHANGE - 12 of 50, 2.485130323748308e-07
Best Score for Price Change Model:2.485130323748308e-07


We can now save each model score in a text file

In [14]:
if not USING_CACHED_DATAFRAME:
    score_price_change_file_name = './output/{}.score.price.change.txt'.format(ASSET_TO_CALCULATE)

    with open(score_price_change_file_name, 'w') as f:
        f.write("{}:{}".format(ASSET_TO_CALCULATE,str(score_price_change)))

We can also calculate for each model the relevant astrological variables, used in each model

In [15]:
if not USING_CACHED_DATAFRAME:
    relevant_features_price_change = sorted( ((v,k) for k,v in booster_price_change.get_score().items()), reverse=True)

    display(relevant_features_price_change)

[(1645.0, 'ASTRO_SUN_POSITION'),
 (1107.0, 'ASTRO_SUN_MOON_DIFF'),
 (519.0, 'ASTRO_MOON_JUPITER_DIFF_ABS'),
 (488.0, 'ASTRO_SUN_MERCURY_DIFF_ABS'),
 (480.0, 'ASTRO_MOON_POSITION'),
 (427.0, 'ASTRO_SUN_MOON_DIFF_ABS'),
 (412.0, 'ASTRO_SUN_VENUS_DIFF_ABS'),
 (343.0, 'ASTRO_MERCURY_MARS_DIFF_ABS'),
 (289.0, 'ASTRO_MOON_VENUS_DIFF_ABS'),
 (269.0, 'ASTRO_MOON_SATURN_DIFF_ABS'),
 (250.0, 'ASTRO_MOON_MERCURY_DIFF'),
 (236.0, 'ASTRO_SUN_SATURN_DIFF_ABS'),
 (227.0, 'ASTRO_MERCURY_SATURN_DIFF_ABS'),
 (215.0, 'ASTRO_VENUS_MERCURY_DIFF_ABS'),
 (213.0, 'ASTRO_MOON_SATURN_DIFF'),
 (208.0, 'ASTRO_SUN_MERCURY_DIFF'),
 (188.0, 'ASTRO_MOON_VENUS_DIFF'),
 (163.0, 'ASTRO_VENUS_POSITION'),
 (163.0, 'ASTRO_SUN_VENUS_DIFF'),
 (140.0, 'ASTRO_SUN_MARS_DIFF_ABS'),
 (136.0, 'ASTRO_SUN_SATURN_DIFF'),
 (136.0, 'ASTRO_MOON_MERCURY_DIFF_ABS'),
 (123.0, 'ASTRO_VENUS_SATURN_DIFF_ABS'),
 (101.0, 'ASTRO_VENUS_MERCURY_DIFF'),
 (94.0, 'ASTRO_MOON_MARS_DIFF'),
 (90.0, 'ASTRO_MERCURY_MARS_DIFF'),
 (89.0, 'ASTRO_SUN_JUPITER_

We can now write such features to text files in order to improve the analisys of the model.

In [16]:
if not USING_CACHED_DATAFRAME:
    def write_features(f, list_to_write):
        for item_to_write in list_to_write:
            f.write('{}-{}'.format(ASSET_TO_CALCULATE,str(item_to_write).replace(')','').replace('(','').replace('\'','').replace(' ','') + '\n'))
        
    features_price_change_file_name = './output/{}.features.price.change.txt'.format(ASSET_TO_CALCULATE)

    with open(features_price_change_file_name, 'w') as f:
        write_features(f,relevant_features_price_change)    

We can check the predicted value for each model on the pandas dataframe, creating a column for it 

In [17]:
if not USING_CACHED_DATAFRAME:
    StockPrices['PredictPriceChange'] = StockPrices.apply(lambda x:predict_score(x, booster_price_change, StockPrices, astro_columns), axis =1)

C:\Users\khali\AppData\Local\Temp\ipykernel_23252\3417710307.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  StockPrices['PredictPriceChange'] = StockPrices.apply(lambda x:predict_score(x, booster_price_change, StockPrices, astro_columns), axis =1)


And save the model for further use on the ```Predict.ipynb``` notebook

In [18]:
booster_price_change.save_model('./output/{}_price_change.model'.format(ASSET_TO_CALCULATE))

C:\Users\khali\AppData\Local\Temp\ipykernel_23252\2170903198.py:1: UserWarning: [11:27:44] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\c_api\c_api.cc:1575: Saving model in the UBJSON format as default.  You can use a file extension: `json` or `ubj` to choose between formats.
  booster_price_change.save_model('./output/{}_price_change.model'.format(ASSET_TO_CALCULATE))


And save a excel with all the data produced...

In [19]:
if not USING_CACHED_DATAFRAME:
    output_excel_file='./output/{}.Analisys.xlsx'.format(ASSET_TO_CALCULATE) 
    StockPrices.to_excel(output_excel_file)

The plotting of charts has been commented out as it is very resource consuming...

In [20]:
"""
swing_to_chart = []
for index, current_swing in StockPrices[StockPrices['PredictSwingTradeScore'] > 0.9].iterrows():
    swing_to_chart.append(dict(
        x0=current_swing['CorrectedDate'], 
        x1=current_swing['CorrectedDate'], 
        y0=0, 
        y1=1, 
        xref='x', 
        yref='paper',
        line_width=2))
"""        

"\nswing_to_chart = []\nfor index, current_swing in StockPrices[StockPrices['PredictSwingTradeScore'] > 0.9].iterrows():\n    swing_to_chart.append(dict(\n        x0=current_swing['CorrectedDate'], \n        x1=current_swing['CorrectedDate'], \n        y0=0, \n        y1=1, \n        xref='x', \n        yref='paper',\n        line_width=2))\n"

In [21]:
"""
fig = go.Figure(data=[go.Candlestick(
                x=StockPrices['CorrectedDate'],
                open=StockPrices['Open'],
                high=StockPrices['High'],
                low=StockPrices['Low'],
                close=StockPrices['Price'])])
fig.update_layout(
    title="{} Swing Trade Opportunities detected by XGBoost".format(ASSET_TO_CALCULATE),
    width=1000,
    height=500,
    xaxis_rangeslider_visible=False,
    shapes=swing_to_chart,
    margin=go.layout.Margin(
        l=0,
        r=0,
        b=0,
        t=30,
        pad=4
    ),    
)
#fig.show()
"""

'\nfig = go.Figure(data=[go.Candlestick(\n                x=StockPrices[\'CorrectedDate\'],\n                open=StockPrices[\'Open\'],\n                high=StockPrices[\'High\'],\n                low=StockPrices[\'Low\'],\n                close=StockPrices[\'Price\'])])\nfig.update_layout(\n    title="{} Swing Trade Opportunities detected by XGBoost".format(ASSET_TO_CALCULATE),\n    width=1000,\n    height=500,\n    xaxis_rangeslider_visible=False,\n    shapes=swing_to_chart,\n    margin=go.layout.Margin(\n        l=0,\n        r=0,\n        b=0,\n        t=30,\n        pad=4\n    ),    \n)\n#fig.show()\n'

In [22]:
"""
swing_to_chart = []
for index, current_swing in StockPrices[StockPrices['PredictPriceIncreaseScore'] > 5].iterrows():
    swing_to_chart.append(dict(
        x0=current_swing['CorrectedDate'], 
        x1=current_swing['CorrectedDate'], 
        y0=0, 
        y1=1, 
        xref='x', 
        yref='paper',
        line_width=2))
"""        

"\nswing_to_chart = []\nfor index, current_swing in StockPrices[StockPrices['PredictPriceIncreaseScore'] > 5].iterrows():\n    swing_to_chart.append(dict(\n        x0=current_swing['CorrectedDate'], \n        x1=current_swing['CorrectedDate'], \n        y0=0, \n        y1=1, \n        xref='x', \n        yref='paper',\n        line_width=2))\n"

In [23]:
"""
fig = go.Figure(data=[go.Candlestick(
                x=StockPrices['CorrectedDate'],
                open=StockPrices['Open'],
                high=StockPrices['High'],
                low=StockPrices['Low'],
                close=StockPrices['Price'])])
fig.update_layout(
    title="{} Price Increase Opportunities detected by XGBoost (Min {}%)".format(ASSET_TO_CALCULATE, STAGNATION_THRESHOLD),
    width=1000,
    height=500,
    xaxis_rangeslider_visible=False,
    shapes=swing_to_chart,
    margin=go.layout.Margin(
        l=0,
        r=0,
        b=0,
        t=30,
        pad=4
    ),    
)
#fig.show()
"""

'\nfig = go.Figure(data=[go.Candlestick(\n                x=StockPrices[\'CorrectedDate\'],\n                open=StockPrices[\'Open\'],\n                high=StockPrices[\'High\'],\n                low=StockPrices[\'Low\'],\n                close=StockPrices[\'Price\'])])\nfig.update_layout(\n    title="{} Price Increase Opportunities detected by XGBoost (Min {}%)".format(ASSET_TO_CALCULATE, STAGNATION_THRESHOLD),\n    width=1000,\n    height=500,\n    xaxis_rangeslider_visible=False,\n    shapes=swing_to_chart,\n    margin=go.layout.Margin(\n        l=0,\n        r=0,\n        b=0,\n        t=30,\n        pad=4\n    ),    \n)\n#fig.show()\n'